# Suite PYS — Python3 simplified by Python's rules

`Python3.Simplifiers.OfAny(term, scope)` simplifies a Python3 term by Python's rules, CPython 3.14's (mbse-basis
[CAP-295: simplification of single-dialect expressions with literal leaves](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-295) to
[CAP-297: simplification in every binding state](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-297)): Python runs
them, and TypeScript models them over JavaScript's values, the same tables of values and errors in both. What the scope
decides folds into the literal of its value (a basic scalar's, or a constant's: `None`, `...`, a complex number), a
list, a tuple, a dict or a non-empty set of such values into its display, a closed constructor expression; a value with
none (an empty set, a slice, a closure, a template, a generator) keeps its subterm, its own subterms simplified; and what Python leaves to a statement, to its
implementation or to its platform stays, named with its class in the evaluation inventory (`residues`): `yield` and
`await` (`d`), `1 is 1` (`b`), a complex power or a locale's format (`a`). What Python makes an error still raises,
by the same class and message. Terms are built through the dialect's writers, in a transaction (`term`); `text` gives
a residual's source, as the printer writes it.

The scope's options choose otherwise for what Python leaves to its implementation or its platform (mbse-basis
[CAP-305: simplification's options, chosen up front](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-305.md),
PYS-15 to PYS-19): a refused format spec raises, equal immutable values are compared as CPython compares them, a
complex power folds by the host's math library, a locale's spec is formatted under a locale given as data, and a field
whose text is the implementation's is written as CPython writes it.

In [ ]:
import codecs
import itertools
import json
import locale
import math
import re
import unicodedata
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.Python.Python3 import Simplifiers as PV, Expressions as P, Text as PText
from mbse.Expressions.Framework import Terms as F
from mbse.Expressions.Framework.Simplifiers import NotEvaluated
from mbse.Schemas.Framework import Builders
from support import raises

store = Builders.Store()  # the framework, the standard dialects Python3's complex numbers use, and the Python family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
                PythonTypes.install, PythonPatterns.install]:
    store.transaction(install)


def term(spec):
    """The Python3 term `spec` writes, a native value as its literal, the root of a holder built within the transaction
    open: the term, a visitor of that transaction, lives within it."""
    return F.term(store.Expressions.Framework.Expression(lambda e: e.root(P.DIALECT.spec(spec))))


def source(spec):
    """The source of the term `spec` writes."""
    return PText.ToText(term(spec))


def simplified(spec, variables=None):
    """The simplification of the term `spec` writes, in a scope binding `variables`."""
    return PV.OfAny(term(spec), PV.Scope(variables or {}))


def value(spec, variables=None):
    """The value the term `spec` writes simplifies to: its literal root's."""
    return simplified(spec, variables).evaluate()


def text(spec, variables=None):
    """The source of the residual of the term `spec` writes, simplified in a scope binding `variables`."""
    return PText.ToText(term(simplified(spec, variables).residual))


def reasons(spec, variables=None):
    """Why what stayed in the residual stayed: each residue's reason and class."""
    return [(residue.reason, residue.category) for residue in simplified(spec, variables).residues]


def folds(rows):
    """Each (expression, variables, value) folds to a literal root of that value, of its type."""
    for spec, variables, expected in rows:
        found = value(spec, variables)
        assert found == expected and type(found) is type(expected), (source(spec), found, expected)


def stays(rows):
    """Each (expression, variables, residual) simplifies to that residual, its root no literal and nothing named: a
    value with no literal, a display, or a term whose subterms simplified."""
    for spec, variables, expected in rows:
        result = simplified(spec, variables)
        shown = PText.ToText(term(result.residual))
        assert not result.literal and not result.residues and shown == expected, (source(spec), shown, expected)


def named(rows):
    """Each (expression, variables, reason, class) leaves one subterm in the residual, named by that reason and class."""
    for spec, variables, reason, category in rows:
        assert reasons(spec, variables) == [(reason, category)], (source(spec), reasons(spec, variables))


def fails(rows):
    """Each (expression, variables, error, message) raises as Python raises it."""
    for spec, variables, error, message in rows:
        with raises(error, match=message):
            simplified(spec, variables)


# the builders of the terms the cases write
n, kw, st, fv, comp = P.name, P.keyword, P.starred, P.formattedvalue, P.comprehension
x, y, z, v, m = n("x"), n("y"), n("z"), n("v"), n("m")
inf, nan = math.inf, math.nan
s = lambda spec: P.call("str", spec)  # noqa: E731: `str(spec)`, which shows what equality cannot (a NaN, a sign)
c1, c2, one = P.constant(1j), P.binop("+", 1, P.constant(2j)), P.constant(1 + 1j)  # 1j, 1 + 2j, (1+1j)


def under(options):
    """The simplification of the term `spec` writes, in a scope binding `variables`, with the options of simplification
    `options` (PYS-15 to PYS-19)."""
    return lambda spec, variables=None: PV.OfAny(term(spec), PV.Scope(variables or {}, options=options))


def refusal(run):
    """The message of the ValueError `run` raises."""
    try:
        run()
    except ValueError as error:
        return str(error)
    raise AssertionError("no ValueError")

<a id="PYS-01"></a>
## PYS-01 · Constants: `None`, `...` and complex numbers fold, and a value is written back as its literal, or as its display

In [ ]:
with store.transaction():
    # `None`, `...` and a complex number are Python3's constants: each folds to its value
    folds([
        (P.constant(None), {}, None), (P.unaryop("not", P.constant(None)), {}, True), (s(P.constant(...)), {}, "Ellipsis"),
        (P.constant(2j), {}, 2j), (lambda u: getattr(u, P.CONSTANT)(lambda k: k.complex(lambda c: c.imag(2.0))), {}, 2j),
        (s(P.constant(1.5j)), {}, "1.5j"), (s(P.constant(2j)), {}, "2j"),
    ])
    assert value(P.constant(...)) is Ellipsis and simplified(P.constant(None)).literal
    fails([
        (lambda u: getattr(u, P.CONSTANT)(lambda r: r), {}, ValueError, "a constant needs a singleton or a complex number"),
        (P.binop("+", P.constant(...), 1), {}, TypeError, "unsupported operand type(s) for +: 'ellipsis' and 'int'"),
    ])
    # A value is written back as its literal, a constant's among them: a literal root
    for bound, written in [(None, "None"), (..., "..."), (1 + 2j, "(1+2j)"), (-2j, "(-0-2j)"), (-1.5, "-1.5"), (-3, "-3"),
                           (True, "True"), ("a", "'a'"), (b"z", "b'z'"), (inf, "float('inf')"), (nan, "float('nan')")]:
        assert text(x, {"x": bound}) == written and simplified(x, {"x": bound}).literal, written
    # A list, a tuple, a dict or a set of values that have literals is written back as its display, a term, not a
    # literal; a value with no literal keeps its term
    stays([
        (P.tuple_(1, P.binop("+", 1, 1)), {}, "(1, 2)"), (P.tuple_(1), {}, "(1,)"), (x, {"x": (1, None, ...)}, "(1, None, ...)"),
        (x, {"x": {(1, 2): [3, (4,)]}}, "{(1, 2): [3, (4,)]}"), (x, {"x": []}, "[]"), (x, {"x": {}}, "{}"),
        (x, {"x": {2, 1}}, "{1, 2}"), (x, {"x": set()}, "x"),  # a non-empty set's display, its items in order (PYS-13)
    ])
    with raises(NotEvaluated, match="the root is not a closed constructor expression: its value has no closed constructor expression"):
        simplified(P.lambda_(1)).evaluate()

<a id="PYS-02"></a>
## PYS-02 · Complex arithmetic as CPython 3.14 computes it: a real operand not made complex, infinities recovered, integer powers, `abs` correctly rounded; a power that needs the platform's functions stays, named

In [ ]:
with store.transaction():
    folds([
        (c2, {}, 1 + 2j), (P.binop("*", c1, c1), {}, -1 + 0j), (P.binop("-", 1.5, c2), {}, 0.5 - 2j),
        (P.binop("/", c2, P.binop("-", 3, P.constant(4j))), {}, -0.2 + 0.4j), (P.binop("/", 1, c2), {}, 0.2 - 0.4j),
        (P.binop("**", P.binop("+", 1, c1), -2), {}, -0.5j), (P.binop("**", P.binop("+", 1, c1), 3), {}, -2 + 2j),
        (P.unaryop("not", P.constant(0j)), {}, True), (P.attribute(c2, "imag"), {}, 2.0),
        (P.compare("==", c1, P.binop("*", c1, 1)), {}, True), (P.compare("==", P.binop("+", 2, P.constant(0j)), 2), {}, True),
        (P.compare("==", 2, P.binop("+", 2, P.constant(0j))), {}, True), (s(P.binop("-", 1, P.constant(2j))), {}, "(1-2j)"),
        (s(P.binop("+", P.constant(complex(1, -0.0)), 1.0)), {}, "(2-0j)"),  # a real is not made complex (3.14)
        (s(P.binop("+", c1, 1.5)), {}, "(1.5+1j)"), (s(P.binop("-", c1, 1.5)), {}, "(-1.5+1j)"), (s(P.binop("*", 2.0, c1)), {}, "2j"),
        (s(P.binop("/", c1, 2.0)), {}, "0.5j"), (s(P.unaryop("-", c2)), {}, "(-1-2j)"), (s(P.unaryop("+", c1)), {}, "1j"),
        (s(P.binop("+", c1, c1)), {}, "2j"), (s(P.binop("-", c1, c1)), {}, "0j"), (s(P.binop("/", 1, P.constant(2 + 1j))), {}, "(0.4-0.2j)"),
        (s(P.binop("/", one, P.constant(4 + 1j))), {}, "(0.29411764705882354+0.17647058823529413j)"),  # Smith's algorithm
        (s(P.binop("**", P.constant(0j), 0.5)), {}, "0j"),
    ])
    # Infinities and NaNs in products and quotients, recovered as C11's Annex G does
    folds([
        (s(P.binop("*", x, y)), {"x": complex(inf, inf), "y": 1 + 0j}, "(inf+infj)"), (s(P.binop("*", y, x)), {"x": complex(inf, inf), "y": 1 + 0j}, "(inf+infj)"),
        (s(P.binop("*", x, y)), {"x": complex(nan, 1e300), "y": 1e300 + 1e300j}, "(-inf+infj)"), (s(P.binop("*", x, one)), {"x": complex(nan, nan)}, "(nan+nanj)"),
        (s(P.binop("/", one, x)), {"x": complex(nan, 1)}, "(nan+nanj)"), (s(P.binop("/", x, one)), {"x": complex(inf, nan)}, "(inf-infj)"),
        (s(P.binop("/", one, x)), {"x": complex(inf, inf)}, "0j"), (s(P.binop("/", 1, x)), {"x": complex(nan, 1)}, "(nan+nanj)"),
        (s(P.binop("/", x, one)), {"x": complex(-inf, nan)}, "(-inf+infj)"), (s(P.binop("/", x, one)), {"x": complex(nan, inf)}, "(inf+infj)"),
        (s(P.binop("/", one, x)), {"x": complex(inf, nan)}, "0j"), (s(P.binop("/", one, x)), {"x": complex(nan, inf)}, "-0j"),
    ])
    # `abs` of a complex number, correctly rounded in both languages
    folds([
        (P.call("abs", P.binop("+", 3, P.constant(4j))), {}, 5.0), (P.call("abs", P.constant(3j)), {}, 3.0),
        (P.call("abs", P.binop("+", x, c1)), {"x": inf}, inf), (s(P.call("abs", P.binop("+", x, c1))), {"x": nan}, "nan"),
        (P.call("abs", x), {"x": complex(5e-324, 5e-324)}, 5e-324),
    ])
    # A complex power whose exponent is no integer, and a real power whose value is complex, need the platform's `pow`,
    # `atan2`, `cos` and `sin`: each stays, named
    named([
        (P.binop("**", c1, 0.5), {}, "Python3 binop '**' of a complex power is not simplified", "a"),
        (P.binop("**", -8, 0.5), {}, "Python3 binop '**' of a complex power is not simplified", "a"),
    ])
    fails([
        (P.binop("/", 1, P.constant(0j)), {}, ZeroDivisionError, "division by zero"),
        (P.binop("/", c1, 0.0), {}, ZeroDivisionError, "division by zero"), (P.binop("/", c1, P.constant(0j)), {}, ZeroDivisionError, "division by zero"),
        (P.binop("**", P.constant(0j), -1), {}, ZeroDivisionError, "zero to a negative or complex power"),
        (P.binop("**", P.constant(0j), c1), {}, ZeroDivisionError, "zero to a negative or complex power"),
        (P.binop("**", P.constant(1e200 + 1e200j), 2), {}, OverflowError, "complex exponentiation"),
        (P.call("abs", P.binop("+", 1.5e308, P.constant(1.5e308j))), {}, OverflowError, "absolute value too large"),
        (P.compare("<", c1, c1), {}, TypeError, "'<' not supported between instances of 'complex' and 'complex'"),
        (P.binop("//", c1, 2), {}, TypeError, "unsupported operand type(s) for //: 'complex' and 'int'"),
    ])

<a id="PYS-03"></a>
## PYS-03 · Parentheses, adjacent strings and named expressions

In [ ]:
with store.transaction():
    folds([
        (P.binop("*", P.parenthesized(P.binop("+", 1, 2)), 3), {}, 9), (P.concatenatedstring("a", "b"), {}, "ab"),
        (P.concatenatedstring(b"a", b"b"), {}, b"ab"), (P.binop("+", P.namedexpr("y", 2), 1), {}, 3),
        (P.namedexpr("y", x), {"x": 1.5}, 1.5),
    ])
    stays([(P.binop("*", P.parenthesized(P.binop("+", x, P.binop("+", 1, 2))), 3), {}, "(x + 3) * 3")])
    fails([
        (P.concatenatedstring(b"a", "b"), {}, TypeError, "cannot mix bytes and nonbytes literals"),
        (P.binop("+", b"a", "b"), {}, TypeError, "can't concat str to bytes"),
    ])

<a id="PYS-04"></a>
## PYS-04 · f-strings: conversions, debug fields, nested format specs and escapes; a residual's fields respelled; a field whose text is an implementation's stays, named

In [ ]:
with store.transaction():
    folds([
        (P.joinedstr(["x=", fv(x), "!"]), {"x": 1.5}, "x=1.5!"), (P.joinedstr([fv(x, "r", (">8",))]), {"x": "a"}, "     'a'"),
        (P.joinedstr([fv(x, None, None, True, "x=")]), {"x": (1, None)}, "x=(1, None)"),
        (P.joinedstr([fv(x, "a")]), {"x": "é"}, "'\\xe9'"), (P.joinedstr([fv(x, "a")]), {"x": "\u2028\U0001F600"}, "'\\u2028\\U0001f600'"),
        (P.joinedstr([fv(x, "a")]), {"x": "α"}, "'\\u03b1'"),
        (P.joinedstr([fv(x, "s")]), {"x": [1.0, None]}, "[1.0, None]"),
        (P.joinedstr([fv(x, None, (fv(n("w")), ".", fv(n("p")), "f"))]), {"x": 3.14159, "w": 8, "p": 3}, "   3.142"),
        (P.joinedstr(["{{", fv(1), "}}\\n\\x41\\u00e9\\101\\q"]), {}, "{1}\nAéA\\q"), (P.joinedstr(["\\n"], "rf"), {}, "\\n"),
        (P.joinedstr([fv(x, "r")]), {"x": {"a": (1,)}}, "{'a': (1,)}"), (P.joinedstr([fv(P.slice_(1, 2), "r")]), {}, "slice(1, 2, None)"),
        (P.joinedstr([]), {}, ""), (P.joinedstr(["a", fv(1)], None), {}, "a1"), (P.joinedstr([fv(x, None, ())]), {"x": 1}, "1"),
        (P.joinedstr([fv(x, None, None, True)]), {"x": 1}, "1"),
    ])
    # A field whose term simplified loses its text as written: a debug field's label is written before it, with `!r` unless
    # it has a conversion or a format spec
    stays([
        (P.joinedstr([fv(P.binop("+", x, P.binop("+", 1, 2)), None, None, True, "x + (1+2)=")]), {}, "f'x + (1+2)={x + 3!r}'"),
        (P.joinedstr([fv(P.binop("+", x, P.binop("+", 1, 2)), None, (">5",), True, "x=")]), {}, "f'x={x + 3:>5}'"),
        (P.joinedstr([fv(P.binop("+", x, P.binop("+", 1, 2)), None, (), False, "x + (1+2)")]), {}, "f'{x + 3:}'"),
        (P.joinedstr([fv(P.binop("+", x, P.binop("+", 1, 2)), None, None, False, "x + (1+2)")]), {}, "f'{x + 3}'"),
        (P.let_("y", 2, P.joinedstr([fv(x), fv(y)])), {}, "f'{x}{2}'"),
    ])
    named([  # a set's text is its hash table's order (`\N{...}`: PYS-14)
        (P.joinedstr([fv(x)]), {"x": {1}}, "Python3 joinedstr's field of set is not simplified", "a"),
        (P.joinedstr([fv(x, "r")]), {"x": {1}}, "Python3 joinedstr's field of set is not simplified", "a"),
    ])

<a id="PYS-05"></a>
## PYS-05 · `format()`'s mini-language for a str, an int, a bool, a float and a complex number, as CPython rounds and composes them; a locale's spec stays, named

In [ ]:
with store.transaction():
    folds([(P.joinedstr([fv(x, None, (spec,))]), {"x": bound}, expected) for bound, spec, expected in [
        (1234, "08,", "0,001,234"), (-1, "06", "-00001"), (1, "=+6", "+    1"), (255, "#010_x", "0x000_00ff"), (65, ">3c", "  A"),
        (12345678, "_b", "1011_1100_0110_0001_0100_1110"), (True, ">5", "    1"), (255, "X", "FF"), (5, "+", "+5"), (5, " ", " 5"),
        (2**60, ".3e", "1.153e+18"), (12345.678, ",.2f", "12,345.68"), (2.675, ".2f", "2.67"), (-0.0, "z.1f", "0.0"),
        (1.5, "#.0f", "2."), (2.5, ".0f", "2"), (1234.5, ".3", "1.23e+03"), (1.0, ".3", "1.0"), (0.5, "%", "50.000000%"),
        (123456789.0, "g", "1.23457e+08"), (100.0, "#g", "100.000"), (1e-5, ".3e", "1.000e-05"), (1e-5, "E", "1.000000E-05"),
        (0.0, "e", "0.000000e+00"), (1e23, ".3e", "1.000e+23"), (9.9999, ".2e", "1.00e+01"), (1.5, "e", "1.500000e+00"),
        (1.5, ">6", "   1.5"), (1.5, " .1f", " 1.5"), (1e16, "", "1e+16"), (-inf, "E", "-INF"), (nan, "f", "nan"),
        (inf, "+f", "+inf"), (nan, "+.1f", "+nan"), ("ab", "^6", "  ab  "), ("ab", "0>5", "000ab"), ("héllo", ".2", "hé"),
        ("a", "*<3", "a**")]])
    # A complex number: each part a float's, the imaginary part signed, then `j`; with no type, a repr's digits (or `g`'s,
    # with a precision), in parentheses unless the real part is +0, which is left out; then padded as a whole
    folds([(P.joinedstr([fv(x, None, (spec,))]), {"x": bound}, expected) for bound, spec, expected in [
        (1 + 2j, ">12", "      (1+2j)"), (1.23456 + 1j, ".3", "(1.23+1j)"), (2j, "+", "+2j"), (1 + 1j, "#", "(1.+1.j)"),
        (1e16 + 1j, "#", "(1.e+16+1.j)"), (1234.5 + 1j, "_.2f", "1_234.50+1.00j"), (1 + 2j, "E", "1.000000E+00+2.000000E+00j"),
        (complex(inf, nan), "F", "INF+NANj"), (complex(-0.0, -0.0), "z.1f", "0.0+0.0j"), (1 + 2j, "-^15", "----(1+2j)-----"),
        (1 + 2j, "*<9", "(1+2j)***"), (1 + 2j, " ", "( 1+2j)"), (-1.5 - 2.5j, "g", "-1.5-2.5j")]])
    named([  # a locale's; a spec Python refuses (zero padding for a complex number); a spec of a value with no format of its own
        (P.joinedstr([fv(x, None, ("n",))]), {"x": 1}, "Python3 joinedstr's format spec 'n' of int is not simplified", "a"),
        (P.joinedstr([fv(c1, None, ("n",))]), {}, "Python3 joinedstr's format spec 'n' of complex is not simplified", "a"),
        (P.joinedstr([fv(c1, None, ("05",))]), {}, "Python3 joinedstr's format spec '05' of complex is not simplified", "a"),
        (P.joinedstr([fv(x, None, (">5",))]), {"x": {1}}, "Python3 joinedstr's format of set is not simplified", "a"),
    ])
    fails([
        (P.joinedstr([fv(P.constant(None), None, (">5",))]), {}, TypeError, "unsupported format string passed to NoneType.__format__"),
        (P.joinedstr([fv(x, None, ("c",))]), {"x": 2**70}, OverflowError, "Python int too large to convert to C long"),
        (P.joinedstr([fv(x, None, ("c",))]), {"x": -1}, OverflowError, "%c arg not in range(0x110000)"),
    ])

<a id="PYS-06"></a>
## PYS-06 · t-strings: a template, as Python 3.14's `string.templatelib` holds it, with no literal; its residual kept as written

In [ ]:
with store.transaction():
    template = P.templatestr(["a", P.interpolation(x, None, None, True, "x = "), "b", P.interpolation(y, "r", (">3",))])
    folds([
        (P.subscript(P.attribute(template, "strings"), 0), {"x": 1, "y": "s"}, "ax = "),
        (P.attribute(P.subscript(P.attribute(template, "interpolations"), 0), "expression"), {"x": 1, "y": "s"}, "x"),
        (P.attribute(P.subscript(P.attribute(template, "interpolations"), 0), "conversion"), {"x": 1, "y": "s"}, "r"),
        (P.attribute(P.subscript(P.attribute(template, "interpolations"), 1), "format_spec"), {"x": 1, "y": "s"}, ">3"),
        (P.subscript(P.attribute(template, "values"), 1), {"x": 1, "y": "s"}, "s"),
        (P.subscript(P.attribute(P.templatestr([]), "strings"), 0), {}, ""),
        (P.subscript(P.attribute(P.templatestr(["a"], None), "strings"), 0), {}, "a"),
        (P.subscript(P.attribute(P.templatestr([P.interpolation(x, None, None, True)]), "strings"), 0), {"x": 1}, ""),
    ])
    stays([(template, {}, "t'a{x = }b{y!r:>3}'"), (template, {"x": 1, "y": 2}, "t'a{x = }b{y!r:>3}'")])  # its text is its value's

<a id="PYS-07"></a>
## PYS-07 · Displays and unpacking: tuples, sets by Python's equality, dicts and their `**` items, `*` in a display or a call, and their operators

In [ ]:
with store.transaction():
    folds([
        (P.compare("==", P.tuple_(1, st(v)), P.tuple_(1, 2, 3)), {"v": [2, 3]}, True), (P.compare("==", P.tuple_(1), P.list_(1)), {}, False),
        (P.call("len", P.set_(1, 1.0, True, 2)), {}, 2), (P.compare("in", 3, P.set_(st(v))), {"v": [3]}, True),
        (P.compare("<", P.set_(1), P.set_(1, 2)), {}, True), (P.call("len", P.binop("^", P.set_(1, 2), P.set_(2, 3))), {}, 2),
        (P.subscript(P.dict_((None, m), ("b", 2)), "a"), {"m": {"a": 1}}, 1), (P.call("max", st(v)), {"v": [1, 3, 2]}, 3),
        (P.compare("<", P.tuple_(1, 2), P.tuple_(1, 2, 0)), {}, True), (P.subscript(P.binop("+", P.tuple_(1), P.tuple_(2)), -1), {}, 2),
        (s(P.set_(1)), {}, "{1}"), (s(P.call("set")), {}, "set()"), (P.unaryop("not", P.tuple_()), {}, True),
        (P.compare("==", P.set_(1, 2), P.set_(2, 1)), {}, True), (P.compare("==", P.dict_(("a", 1)), m), {"m": {"a": 1}}, True),
        (P.compare("==", P.dict_(("a", 1)), P.dict_(("b", 1))), {}, False), (P.compare("<", P.list_(1, 2), P.list_(1, 3)), {}, True),
        (P.compare("<=", P.set_(1), P.set_(1)), {}, True), (P.compare(">", P.set_(1, 2), P.set_(1)), {}, True),
        (P.compare(">=", P.set_(1), P.set_(1, 2)), {}, False), (P.call("len", P.binop("*", P.tuple_(1, 2), 2)), {}, 4),
        (P.call("len", P.binop("+", P.list_(1), P.list_(2))), {}, 2), (P.call("len", P.binop("*", P.list_(1), 2)), {}, 2),
        (P.subscript(P.dict_((P.tuple_(1, 2), 3)), P.tuple_(1, 2)), {}, 3), (P.call("len", P.binop("&", P.set_(1, 2), P.set_(2, 3))), {}, 1),
        (P.call("len", P.binop("|", P.set_(1), P.set_(2))), {}, 2), (P.subscript(P.binop("|", P.dict_(("a", 1)), P.dict_(("b", 2))), "b"), {}, 2),
        (P.call("len", P.binop("-", P.set_(1, 2), P.set_(2))), {}, 1),
    ])
    stays([
        (P.set_(P.binop("+", 1, 1), y), {}, "{2, y}"), (P.list_(st(v), 3), {"v": (1, 2)}, "[1, 2, 3]"), (P.list_(st(v)), {}, "[*v]"),
        (P.dict_((None, m), ("b", P.binop("+", 1, 1))), {"m": {"a": 1}}, "{'a': 1, 'b': 2}"), (P.dict_((None, m), ("b", 2)), {}, "{**m, 'b': 2}"),
    ])
    fails([
        (P.list_(st(v)), {"v": 5}, TypeError, "Value after * must be an iterable, not int"),
        (P.dict_((None, m)), {"m": 1}, TypeError, "'int' object is not a mapping"),
        (P.set_(v), {"v": [1]}, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"),
        (P.dict_((P.tuple_(1, P.list_()), 2)), {}, TypeError, "cannot use 'tuple' as a dict key (unhashable type: 'list')"),
        (P.binop("+", P.list_(1), P.tuple_(2)), {}, TypeError, 'can only concatenate list (not "tuple") to list'),
        (P.list_(st(P.generatorexp(P.binop("/", 1, n("p")), comp("p", v)))), {"v": [0]}, ZeroDivisionError, "division by zero"),
    ])

<a id="PYS-08"></a>
## PYS-08 · Slices and subscripts by one: negative indices, steps and bounds clamped, as Python slices a sequence

In [ ]:
with store.transaction():
    folds([
        (P.subscript("abcde", P.slice_(1, -1)), {}, "bcd"), (P.subscript("abcde", P.slice_(None, None, -2)), {}, "eca"),
        (P.subscript(b"abc", P.slice_(None, None, 2)), {}, b"ac"), (P.call("len", P.subscript(v, P.slice_(5, 1, -2))), {"v": [1] * 6}, 2),
        (P.attribute(P.slice_(1, 2), "step"), {}, None), (P.call("len", P.subscript(P.list_(1, 2, 3), P.slice_(-10, None, -1))), {}, 0),
        (P.subscript(P.subscript(P.list_(1, 2, 3), P.slice_(10, None, -1)), 0), {}, 3), (P.call("len", P.subscript(P.list_(1, 2, 3), P.slice_(-10))), {}, 3),
        (P.subscript(P.subscript(P.tuple_(1, 2, 3), P.slice_(1)), 0), {}, 2), (P.call("len", P.subscript(P.list_(1, 2, 3), P.slice_(10))), {}, 0),
    ])
    stays([(P.subscript(v, P.slice_(1)), {"v": [1, 2, 3]}, "[2, 3]"), (P.subscript(v, P.slice_(P.binop("+", 0, 1))), {}, "v[1:]"),
           (P.slice_(1, 2), {}, "1:2")])  # a slice has no literal
    fails([
        (P.subscript(v, P.slice_(None, None, 0)), {"v": [1]}, ValueError, "slice step cannot be zero"),
        (P.subscript(v, P.slice_(1.5)), {"v": [1]}, TypeError, "slice indices must be integers or None or have an __index__ method"),
        (P.subscript(P.tuple_(1, 2), 5), {}, IndexError, "tuple index out of range"),
    ])

<a id="PYS-09"></a>
## PYS-09 · Comprehensions: a list's, a set's and a dict's, of several clauses, over targets that unpack; a target that writes into data stays, named

In [ ]:
with store.transaction():
    folds([
        (P.call("sum", P.listcomp(P.binop("*", n("p"), n("q")), comp(P.tuple_(n("p"), n("q")), v))), {"v": [(1, 2), (3, 4)]}, 14),
        (P.call("len", P.setcomp(P.binop("%", n("p"), 2), comp("p", v))), {"v": [1, 2, 3]}, 2),
        (P.subscript(P.dictcomp(n("p"), P.binop("*", n("p"), n("p")), comp("p", v, (P.compare(">", n("p"), 1),))), 3), {"v": [1, 2, 3]}, 9),
        (P.call("sum", P.generatorexp(P.call("len", n("q")), comp(P.tuple_(n("p"), st(n("q"))), v))), {"v": [[1, 2, 3], [4]]}, 2),
        (P.call("sum", P.listcomp(n("p"), comp(P.parenthesized(n("p")), v))), {"v": [1, 2]}, 3),
        (P.call("sum", P.listcomp(n("q"), comp("p", v), comp("q", n("p")))), {"v": [[1, 2], [3]]}, 6),
        (P.call("sum", P.generatorexp(st(n("p")), comp("p", v))), {"v": [[1, 2], [3]]}, 6),  # `(*p for p in v)`, Python 3.15
        (P.subscript(P.dictcomp(m, None, comp("m", v)), "b"), {"v": [{"a": 1}, {"b": 2}]}, 2),  # `{**m for m in v}`, Python 3.15
    ])
    stays([
        (P.listcomp(P.binop("*", n("p"), 2), comp("p", v)), {"v": [1, 2]}, "[2, 4]"),
        (P.listcomp(P.binop("*", n("p"), P.binop("+", 1, 1)), comp("p", v)), {}, "[p * 2 for p in v]"),
        (P.dictcomp(n("p"), 0, comp("p", "ab")), {}, "{'a': 0, 'b': 0}"), (P.setcomp(n("p"), comp("p", v)), {"v": [1]}, "{1}"),
    ])
    named([  # a target that writes into data, and an `async for`, need a statement (d); two starred targets are no syntax (b)
        (P.listcomp(n("p"), comp(P.tuple_(n("p"), P.subscript(n("o"), 0)), v)), {"v": [1]}, "Python3 subscript target is not simplified", "d"),
        (P.listcomp(n("p"), comp(P.attribute(n("o"), "a"), v)), {"v": [1]}, "Python3 attribute target is not simplified", "d"),
        (P.listcomp(n("p"), comp("p", v, (), True)), {"v": [1]}, "Python3 async comprehension is not simplified", "d"),
        (P.listcomp(n("p"), comp(P.tuple_(st(n("p")), st(n("q"))), v)), {"v": [1]}, "Python3 starred target is not simplified", "b"),
    ])
    fails([
        (P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), {"v": [1]}, TypeError, "cannot unpack non-iterable int object"),
        (P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), {"v": [[1, 2, 3]]}, ValueError, "too many values to unpack (expected 2, got 3)"),
        (P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), {"v": [[1]]}, ValueError, "not enough values to unpack (expected 2, got 1)"),
        (P.listcomp(n("p"), comp(P.tuple_(n("p"), st(n("q")), n("r")), v)), {"v": [[1]]}, ValueError,
         "not enough values to unpack (expected at least 2, got 1)"),
    ])

<a id="PYS-10"></a>
## PYS-10 · Lambdas and keywords: closures binding Python's parameters, with CPython's messages, and the builtins' keywords

In [ ]:
with store.transaction():
    folds([
        (P.call(P.lambda_(P.binop("+", n("a"), n("b")), ["a", P.arg("b", 10)]), 1), {}, 11),
        (P.call(P.lambda_(n("k"), kwonlyargs=["k"]), kw("k", 4)), {}, 4), (P.call(P.lambda_(P.call("len", n("r")), vararg="r"), 1, 2, 3), {}, 3),
        (P.call(P.lambda_(P.subscript(n("o"), "z"), kwarg="o"), kw(None, m)), {"m": {"z": 5}}, 5),
        (P.call(P.lambda_(n("a"), posonlyargs=["a"]), 7), {}, 7), (P.call(P.lambda_(n("a"), ["a"]), st(v)), {"v": [8]}, 8),
        (P.call("max", v, kw("key", P.lambda_(P.unaryop("-", x), ["x"]))), {"v": [3, 1, 2]}, 1),
        (P.call("min", v, kw("default", 0)), {"v": []}, 0), (P.call("sum", v, kw("start", 10)), {"v": [1, 2]}, 13),
    ])
    # A closure has no literal: a lambda stays, its body simplified in the scope it is made in
    stays([
        (P.lambda_(P.binop("+", x, P.binop("+", 1, 2)), ["x"]), {}, "lambda x: x + 3"),
        (P.lambda_(P.binop("+", x, y), ["x"]), {"y": 2}, "lambda x: x + 2"), (P.call(P.lambda_(x, ["x"]), z), {}, "(lambda x: x)(z)"),
    ])
    fails([
        (P.call(P.lambda_(n("a"), ["a"])), {}, TypeError, "<lambda>() missing 1 required positional argument: 'a'"),
        (P.call(P.lambda_(n("a"), ["a", "b", "c"])), {}, TypeError, "<lambda>() missing 3 required positional arguments: 'a', 'b', and 'c'"),
        (P.call(P.lambda_(n("a"), ["a", "b"]), 1, 2, 3), {}, TypeError, "<lambda>() takes 2 positional arguments but 3 were given"),
        (P.call(P.lambda_(n("a"), ["a", P.arg("b", 1)]), 1, 2, 3), {}, TypeError, "<lambda>() takes from 1 to 2 positional arguments but 3 were given"),
        (P.call(P.lambda_(0), 1), {}, TypeError, "<lambda>() takes 0 positional arguments but 1 was given"),
        (P.call(P.lambda_(n("a"), ["a"], kwonlyargs=["k"]), 1, 2, kw("k", 3)), {}, TypeError,
         "<lambda>() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given"),
        (P.call(P.lambda_(0, kwonlyargs=["k", "j"]), 1, kw("k", 1), kw("j", 2)), {}, TypeError,
         "<lambda>() takes 0 positional arguments but 1 positional argument (and 2 keyword-only arguments) were given"),
        (P.call(P.lambda_(n("a"), ["a"]), 1, kw("a", 2)), {}, TypeError, "<lambda>() got multiple values for argument 'a'"),
        (P.call(P.lambda_(n("a"), ["a"]), kw("b", 2)), {}, TypeError, "<lambda>() got an unexpected keyword argument 'b'"),
        (P.call(P.lambda_(n("k"), kwonlyargs=["k"])), {}, TypeError, "<lambda>() missing 1 required keyword-only argument: 'k'"),
        (P.call(P.lambda_(n("a"), posonlyargs=["a"]), kw("a", 1)), {}, TypeError,
         "<lambda>() got some positional-only arguments passed as keyword arguments: 'a'"),
        (P.call(P.lambda_(n("a"), ["a"]), kw("a", 1), kw(None, m)), {"m": {"a": 2}}, TypeError, "<lambda>() got multiple values for keyword argument 'a'"),
        (P.call(P.lambda_(n("a"), ["a"]), kw(None, m)), {"m": {1: 5}}, TypeError, "keywords must be strings"),
        (P.call(P.lambda_(n("a"), ["a"]), kw(None, m)), {"m": [1]}, TypeError, "<lambda>() argument after ** must be a mapping, not list"),
        (P.call(P.lambda_(n("a"), ["a"]), st(v)), {"v": 5}, TypeError, "<lambda>() argument after * must be an iterable, not int"),
        (P.call("max", v, kw("foo", 1)), {"v": [1]}, TypeError, "max() got an unexpected keyword argument 'foo'"),
        (P.call("min", 1, 2, kw("default", 0)), {}, TypeError, "Cannot specify a default for min() with multiple positional arguments"),
        (P.call("abs", kw("x", 1)), {}, TypeError, "abs() takes no keyword arguments"),
    ])
    # A function the scope provides takes keywords as Python's do (in TypeScript, none: a JavaScript function has no
    # keyword parameters)
    assert value(P.call(n("f"), kw("k", 1)), {"f": lambda **k: k["k"]}) == 1

<a id="PYS-11"></a>
## PYS-11 · `is` and `is not` by identity, and `@`: whether equal immutable values are one object Python leaves to its implementation, which stays, named

In [ ]:
with store.transaction():
    folds([
        (P.compare("is", x, P.constant(None)), {"x": None}, True), (P.compare("is not", 1, P.constant(None)), {}, True),
        (P.compare("is", x, x), {"x": [1]}, True), (P.compare("is", P.list_(1), P.list_(1)), {}, False),
        (P.compare("is", 1, 1.0), {}, False), (P.compare("is", True, True), {}, True), (P.compare("is", 1, 2), {}, False),
        (P.compare("is", P.constant(...), P.constant(...)), {}, True), (P.compare("is not", x, y), {"x": "a", "y": "b"}, True),
    ])
    named([
        (P.compare("is", 1, 1), {}, "Python3 compare 'is' of equal immutable values is not simplified", "b"),
        (P.compare("is", x, y), {"x": nan, "y": nan}, "Python3 compare 'is' of equal immutable values is not simplified", "b"),
        (P.compare("is not", x, y), {"x": "a", "y": "a"}, "Python3 compare 'is' of equal immutable values is not simplified", "b"),
    ])
    # `@` calls an operand's `__matmul__`, which no builtin type has
    fails([(P.binop("@", x, y), {"x": [1], "y": [2]}, TypeError, "unsupported operand type(s) for @: 'list' and 'list'")])

<a id="PYS-12"></a>
## PYS-12 · What stays, by its class: `yield`, `yield from` and `await` (d), invalid syntax and a kind of the family's other dialects (b); `FOLDED`

In [ ]:
with store.transaction():
    named([
        (P.yield_(1), {}, "Python3 yield is not simplified", "d"), (P.yieldfrom(v), {"v": [1]}, "Python3 yieldfrom is not simplified", "d"),
        (P.await_(x), {"x": 1}, "Python3 await is not simplified", "d"),
        (lambda u: u.missing(lambda r: r), {}, "Python3 missing is not simplified", "b"),
    ])
    # What stays keeps its subterms simplified, and what does not depend on it folds around it
    assert text(P.binop("+", P.await_(P.binop("+", 1, 2)), 1)) == "await 3 + 1"
    assert reasons(P.binop("+", P.await_(P.binop("+", 1, 2)), 1)) == [("Python3 await is not simplified", "d")]
    assert text(P.boolop("or", True, P.yield_(1))) == "True"
    # A kind of Python Types, Python Patterns or Python RegExp is its own dialect's, of class (b): simplification across a
    # sub-dialect (mbse-basis CAP-299) is to reach it
    assert [PV.SIMPLIFIER.category(dialect, "typevar", "typevar") for dialect in ("Python Types", "Python Patterns", "Python RegExp")] == ["b"] * 3
    assert PV.SIMPLIFIER.category("Python3", "binop '**' of a complex power", "binop") == "a"
    assert PV.FOLDED == {"literal", "constant", "variable", "let", "list", "tuple", "set", "dict", "starred", "parenthesized",
                         "concatenatedstring", "joinedstr", "templatestr", "attribute", "subscript", "slice", "call", "compare",
                         "boolop", "binop", "unaryop", "ifexp", "namedexpr", "lambda", "listcomp", "setcomp", "dictcomp",
                         "generatorexp", "import", "importfrom"}
    # What Python may never evaluate stays, though it would raise: a branch, by an unknown test; what follows an unknown
    # operand of `and` or `or`, or a chain's first comparison; a lambda's body
    div = P.binop("/", 1, 0)
    for spec, written in [(P.ifexp(x, div, 2), "1 / 0 if x else 2"), (P.boolop("and", x, div), "x and 1 / 0"),
                          (P.compare("<", x, 1, ("<", div)), "x < 1 < 1 / 0"), (P.lambda_(div), "lambda: 1 / 0")]:
        assert text(spec) == written and reasons(spec) == [], text(spec)
    with raises(ZeroDivisionError, match="division by zero"):  # taken, it raises
        text(P.ifexp(True, div, 2))

<a id="PYS-13"></a>
## PYS-13 · Sets: a non-empty set written as its display, its items in an order both implementations compute; an empty set stays

In [ ]:
with store.transaction():
    # A non-empty set is written as its display, a closed constructor expression, its items in an order of their values
    # that both implementations compute, not its hash table's: None, `...`, real numbers (a NaN after every other), complex
    # numbers (by their real parts, then their imaginary ones), strs and bytes as Python orders them, then tuples, item by
    # item
    stays([
        (x, {"x": {10, 9, 2.5, True}}, "{True, 2.5, 9, 10}"), (x, {"x": {"b", "a", "é", "B"}}, "{'B', 'a', 'b', 'é'}"),
        (x, {"x": {None, ..., 1j, 1, "a", b"z", (1, 2), (1,), (0, "z")}}, "{None, ..., 1, 1j, 'a', b'z', (0, 'z'), (1,), (1, 2)}"),
        (x, {"x": {nan, -inf, 1}}, "{-float('inf'), 1, float('nan')}"), (x, {"x": {b"b", b"a"}}, "{b'a', b'b'}"),
        (x, {"x": {complex(nan, 1), 1 + 2j, 2j, 1j}}, "{1j, 2j, (1+2j), (nan+1j)}"),
        (x, {"x": {(None, 1), (None, 0), (..., 0)}}, "{(None, 0), (None, 1), (..., 0)}"),
        (P.set_(3, 1, P.binop("+", 1, 1)), {}, "{1, 2, 3}"), (P.setcomp(P.binop("%", n("p"), 3), comp("p", v)), {"v": [5, 4]}, "{1, 2}"),
        (x, {"x": [1, {2}]}, "[1, {2}]"), (x, {"x": {"k": {2, 1}}}, "{'k': {1, 2}}"),
    ])
    # Its root a closed constructor expression, not a literal, the simplification evaluated the set
    assert simplified(x, {"x": {2, 1}}).constructed and not simplified(x, {"x": {2, 1}}).literal
    assert value(P.set_(2, 1)) == {1, 2} and type(value(P.set_(2, 1))) is set
    # The framework's `known`, which a dialect's `simplify` hook calls with a value it computed, writes a set's display too
    known = PV.SIMPLIFIER.known(term(x), {2, 1})
    assert known.constructed and not known.literal and PText.ToText(term(known.expression)) == "{1, 2}"
    # An empty set has no display (`{}` is a dict, and `set()` calls a name), nor has a set of an item with none: each stays
    stays([(x, {"x": set()}, "x"), (P.call("set"), {}, "set()"), (x, {"x": [set()]}, "x"),
           (P.set_(P.lambda_(1)), {}, "{lambda: 1}")])
    with raises(NotEvaluated, match="the root is not a closed constructor expression: its value has no closed constructor expression"):
        simplified(P.binop("-", P.set_(1), P.set_(1))).evaluate()  # its value an empty set

<a id="PYS-14"></a>
## PYS-14 · `\N{...}` by Unicode's character names and aliases, as CPython 3.14 resolves them, kept as data for TypeScript; a malformed escape and an unknown name stay, named (b)

In [ ]:
with store.transaction():
    # `\N{...}` names a character by a name or an alias of Unicode's, its case ignored, as CPython 3.14's compiler reads it:
    # Hangul's syllables and the ideographs' names by rule, but no named sequence
    folds([
        (P.joinedstr(["\\N{BULLET}"]), {}, "\u2022"), (P.joinedstr(["\\N{bullet}\\N{Latin Small Letter A}"]), {}, "\u2022a"),
        (P.joinedstr(["\\N{LF}\\N{LINE FEED}\\N{BYTE ORDER MARK}"]), {}, "\n\n\ufeff"),  # aliases
        (P.joinedstr(["\\N{HANGUL SYLLABLE GAG}\\N{hangul syllable a}"]), {}, "\uac01\uc544"),
        (P.joinedstr(["\\N{CJK UNIFIED IDEOGRAPH-4E00}\\N{tangut ideograph-17000}\\N{CJK COMPATIBILITY IDEOGRAPH-2F800}"]), {},
         "\u4e00\U00017000\U0002f800"),
        (P.joinedstr(["a\\N{BULLET}}}b", fv(1)]), {}, "a\u2022}b1"), (P.joinedstr(["\\N{BULLET}"], "rf"), {}, "\\N{BULLET}"),
        (P.call("len", P.joinedstr(["\\N{ARABIC LETTER TEH}\\N{ARABIC LETTER TEH MARBUTA}\\N{SEGMENTED DIGIT NINE}"])), {}, 3),
    ])
    assert text(P.joinedstr(["\\N{BULLET}"])) == "'\u2022'" and simplified(P.joinedstr(["\\N{BULLET}"])).literal
    # What Python refuses to compile, invalid syntax (b): an unknown name (a named sequence's, a non-ASCII one, an
    # ideograph's beyond its range or not in four uppercase hexadecimal digits at least), and a malformed escape
    named([(P.joinedstr([spelling]), {}, "Python3 joinedstr's unknown Unicode character name is not simplified", "b") for spelling in [
        "\\N{NO SUCH NAME}", "\\N{LATIN CAPITAL LETTER A WITH MACRON AND GRAVE}", "\\N{\u00e9}", "\\N{CJK UNIFIED IDEOGRAPH-04E00}",
        "\\N{CJK UNIFIED IDEOGRAPH-4E0G}", "\\N{CJK UNIFIED IDEOGRAPH-3300}", "\\N{CJK UNIFIED IDEOGRAPH-A000}"]])
    named([(P.joinedstr([spelling]), {}, "Python3 joinedstr's malformed \\N character escape is not simplified", "b")
           for spelling in ["\\N", "\\NBULLET", "\\N{}", "\\N{BULLET"]])
    # The names are kept as data, `grammars/python3-unicode-names.json`, which TypeScript reads, Python reading its own
    # `unicodedata`: Unicode 16.0's, 45,786 names, and 19 ranges of the ideographs' names by rule
    data = json.loads(Path("../../grammars/python3-unicode-names.json").read_text(encoding="utf-8"))
    held = {}
    for alias in data["aliases"]:
        code, name = alias.split(";")
        held[name] = int(code, 16)
    hangul, point = data["hangul"], int(data["hangul"]["first"], 16)
    for initial, medial, final in itertools.product(hangul["initial"], hangul["medial"], hangul["final"]):
        held[hangul["prefix"] + initial + medial + final], point = point, point + 1
    previous, point = [], -1
    for run in data["characters"]:  # a run of names that differ in their last word alone
        delta, shared, rest = run.split(";")
        first, *lasts = rest.split("|")
        words = previous[:int(shared)] + first.split(" ")
        point += int(delta, 16) if delta else 1
        held[" ".join(words)] = point
        for last in lasts:
            words[-1], point = last, point + 1
            held[" ".join(words)] = point
        previous = words
    assert data["unicode"] == "16.0.0" and len(held) == 45786 and len(data["ranges"]) == 19
    # Python checks the data against its own: every name it holds resolves to its code point, every name of a range, and
    # none beside a range; every name `unicodedata` gives a character is among them
    assert unicodedata.unidata_version == data["unicode"]


    def escape(name):
        """What `\\N{name}` reads as, as CPython's compiler reads it, or None."""
        try:
            return codecs.decode(("\\N{" + name + "}").encode("ascii"), "unicode_escape")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return None


    assert all(escape(name) == chr(code) for name, code in held.items())
    ranged = {}
    for r in data["ranges"]:
        first, last = int(r["first"], 16), int(r["last"], 16)
        assert escape(f"{r['prefix']}{first - 1:04X}") is None and escape(f"{r['prefix']}{last + 1:04X}") is None
        ranged.update({code: f"{r['prefix']}{code:04X}" for code in range(first, last + 1)})
    assert all(escape(name) == chr(code) for code, name in ranged.items())
    for code in range(0x110000):
        name = unicodedata.name(chr(code), None)
        assert name is None or held.get(name) == code or ranged.get(code) == name, (hex(code), name)

<a id="PYS-15"></a>
## PYS-15 · A format spec Python refuses stays, named; the scope's option `specs` "raise" raises Python's `ValueError`, with CPython's message

In [ ]:
with store.transaction():
    # The options of the scope that take a string, each with its values, its default first; the simplifier refuses, before
    # it simplifies, a value an option does not take
    assert PV.OPTIONS == {"specs": ("stay", "raise"), "identity": ("stay", "cpython"), "math": ("stay", "host"),
                          "text": ("stay", "cpython")}
    with raises(ValueError, match="the option 'specs' is 'stay' or 'raise', not 'strict'"):
        under({"specs": "strict"})(x)
    assert PV.OfAny(term(x), {"x": 1}).literal and PV.SIMPLIFIER(term(1)).literal  # a mapping, or none: no options
    # A spec Python refuses stays, named; under "raise" it raises Python's ValueError, with CPython's message, which the
    # TypeScript implementation models (Python checks each message against its own `format`)
    raising = under({"specs": "raise"})
    for bound, spec, owner, message in [
        (1, "abc", "int", "Invalid format specifier 'abc' for object of type 'int'"),
        (1.5, ",_", "float", "Cannot specify both ',' and '_'."), (1.5, "_,", "float", "Cannot specify both ',' and '_'."),
        (1.5, ".,_f", "float", "Cannot specify both ',' and '_'."), (1.5, "._,f", "float", "Cannot specify both ',' and '_'."),
        (1, ".", "int", "Format specifier missing precision"),
        (1, "99999999999999999999", "int", "Too many decimal digits in format string"),
        (1.5, ".99999999999999999999f", "float", "Too many decimal digits in format string"),
        (1, ",c", "int", "Cannot specify ',' with 'c'."), (1, ",x", "int", "Cannot specify ',' with 'x'."),
        ("ab", ",", "str", "Cannot specify ',' with 's'."), (1, ".2,n", "int", "Cannot specify ',' with 'n'."),
        (1, "_\x01", "int", "Cannot specify '_' with '\\x1'."),
        (1, "é", "int", "Unknown format code '\\xe9' for object of type 'int'"),
        (True, "s", "bool", "Unknown format code 's' for object of type 'bool'"),
        (1.5, "d", "float", "Unknown format code 'd' for object of type 'float'"),
        ("ab", "d", "str", "Unknown format code 'd' for object of type 'str'"),
        ("ab", "+s", "str", "Sign not allowed in string format specifier"), ("ab", " ", "str", "Space not allowed in string format specifier"),
        ("ab", "z", "str", "Negative zero coercion (z) not allowed in string format specifier"),
        ("ab", "#", "str", "Alternate form (#) not allowed in string format specifier"),
        ("ab", "=5", "str", "'=' alignment not allowed in string format specifier"),
        (1, ".2d", "int", "Precision not allowed in integer format specifier"),
        (1, "zd", "int", "Negative zero coercion (z) not allowed in integer format specifier"),
        (65, "+c", "int", "Sign not allowed with integer format specifier 'c'"),
        (65, "#c", "int", "Alternate form (#) not allowed with integer format specifier 'c'"),
        (1.5, ".2147483648f", "float", "precision too big"), (1 + 2j, ".2147483648", "complex", "precision too big"),
        (1 + 2j, "%", "complex", "Unknown format code '%' for object of type 'complex'"),
        (1 + 2j, "05", "complex", "Zero padding is not allowed in complex format specifier"),
        (1 + 2j, "=5", "complex", "'=' alignment flag is not allowed in complex format specifier"),
    ]:
        field = P.joinedstr([fv(x, None, (spec,))])
        assert refusal(lambda: format(bound, spec)) == message, spec  # CPython's own
        assert refusal(lambda: raising(field, {"x": bound})) == message, spec
        assert reasons(field, {"x": bound}) == [(f"Python3 joinedstr's format spec {spec!r} of {owner} is not simplified", "a")], spec
    # A spec Python accepts but the simplifier does not format stays under "raise" too: a fraction's digits grouped (3.14),
    # a locale's (PYS-18); a width in another script's digits folds, and `#` writes a point before an exponent
    for spec in [".3,f", ".,", "n"]:
        assert [r.reason for r in raising(P.joinedstr([fv(x, None, (spec,))]), {"x": 1.5}).residues] == [
            f"Python3 joinedstr's format spec {spec!r} of float is not simplified"]
    folds([(P.joinedstr([fv(x, None, ("٣",))]), {"x": 1}, "  1"), (P.joinedstr([fv(x, None, ("#",))]), {"x": 1e16}, "1.e+16")])

<a id="PYS-16"></a>
## PYS-16 · `is` between equal immutable values: the scope's option `identity` "cpython" answers as CPython does where the values decide it, for the values it keeps one object of; any other pair stays, named

In [ ]:
with store.transaction():
    # Under "cpython", CPython 3.14's answer where the values decide it: True for its small ints (-5 to 256) and the empty
    # tuple, str and bytes, which it keeps one object of, however they are made; a closure's scope keeps the options
    cpython = under({"identity": "cpython"})
    for spec, variables in [(P.compare("is", 1, 1), {}), (P.compare("is", P.unaryop("-", 5), x), {"x": -5}),
                            (P.compare("is", 256, x), {"x": 256}), (P.compare("is", P.tuple_(), P.tuple_()), {}),
                            (P.compare("is", "", x), {"x": ""}), (P.compare("is", b"", b""), {}),
                            (P.call(P.lambda_(P.compare("is", n("a"), 1), ["a"]), 1), {})]:
        assert cpython(spec, variables).evaluate() is True, source(spec)
    assert cpython(P.compare("is not", x, y), {"x": 7, "y": 7}).evaluate() is False
    # Whether any other pair of equal immutable values is one object depends on how they were made: it stays, named, as by
    # default (PYS-11)
    for spec, variables in [(P.compare("is", 257, x), {"x": 257}), (P.compare("is", x, -6), {"x": -6}), (P.compare("is", "a", "a"), {}),
                            (P.compare("is", 1.0, 1.0), {}), (P.compare("is", P.tuple_(1), P.tuple_(1)), {}),
                            (P.compare("is", x, y), {"x": nan, "y": nan})]:
        assert [(r.reason, r.category) for r in cpython(spec, variables).residues] == [
            ("Python3 compare 'is' of equal immutable values is not simplified", "b")], source(spec)
    with raises(ValueError, match="the option 'identity' is 'stay' or 'cpython', not 'yes'"):
        under({"identity": "yes"})(x)

<a id="PYS-17"></a>
## PYS-17 · A complex power that needs the platform's functions: the scope's option `math` "host" folds it by the host's math library, as CPython's `_Py_c_pow` computes it

In [ ]:
with store.transaction():
    # Under "host", a complex power whose exponent is no integer, and a real power whose value is complex, fold by the host's
    # `hypot`, `pow`, `atan2`, `exp`, `log`, `cos` and `sin`, each host's last bit its own: the cases assert exact results
    host = under({"math": "host"})
    for spec, written in [(s(P.binop("**", -1, 0.5)), "(6.123233995736766e-17+1j)"),
                          (s(P.binop("**", P.binop("+", 4, P.constant(0j)), 0.5)), "(2+0j)"),
                          (s(P.binop("**", P.binop("+", 1, P.constant(0j)), c1)), "(1+0j)")]:
        assert host(spec).evaluate() == written, source(spec)
    with raises(OverflowError, match="complex exponentiation"):
        host(P.binop("**", P.constant(1e300 + 1e300j), 2.5))
    # By default they stay, named (PYS-02); a negative infinity to a fractional power is a real infinity, as Python's is
    assert [r.reason for r in under({})(P.binop("**", -1, 0.5)).residues] == ["Python3 binop '**' of a complex power is not simplified"]
    folds([(P.binop("**", x, 0.5), {"x": -inf}, inf)])
    with raises(ValueError, match="the option 'math' is 'stay' or 'host', not 'libm'"):
        under({"math": "libm"})(x)

<a id="PYS-18"></a>
## PYS-18 · A format spec a locale decides (`n`): the scope's option `locale`, a locale as `locale.localeconv()` gives one, formats it under that locale, as CPython does

In [ ]:
with store.transaction():
    # A locale is data: its decimal point, its thousands separator and its grouping, each group's size from the right, 0
    # repeating the last and 127 (C's CHAR_MAX) ending the grouping; an int is written as `d` writes it, a float and each
    # part of a complex number as `g` writes them, their integer digits grouped
    de = {"decimal_point": ",", "thousands_sep": ".", "grouping": [3, 0]}
    c = {"decimal_point": ".", "thousands_sep": "", "grouping": []}
    india = {"decimal_point": ".", "thousands_sep": ",", "grouping": [3, 2, 0]}
    once = {"decimal_point": ".", "thousands_sep": "'", "grouping": [1, 127]}
    for given, bound, spec, written in [
        (de, 1234567, "n", "1.234.567"), (de, 1234567.25, "n", "1,23457e+06"), (de, 12345.678, ".7n", "12.345,68"),
        (de, 1234, "012n", "0.000.001.234"), (de, -1234.5, "+n", "-1.234,5"), (de, 1234.5, "*^11n", "**1.234,5**"),
        (de, 1234.5, "#.6n", "1.234,50"), (de, True, "n", "1"), (de, 1234.5 + 2j, "n", "1.234,5+2j"),
        (de, 1234.5 + 2j, ">14n", "    1.234,5+2j"), (de, 1234, "<8n", "1.234   "), (de, -1234, "*=9n", "-***1.234"),
        (c, 1234567, "n", "1234567"), (c, 1234.5678, "n", "1234.57"), (c, -0.0, "zn", "0"), (c, inf, "08n", "00000inf"),
        (india, 123456789, "n", "12,34,56,789"), (once, 12345, "n", "1234'5"),
    ]:
        assert under({"locale": given})(P.joinedstr([fv(x, None, (spec,))]), {"x": bound}).evaluate() == written, (bound, spec)
    # Python checks the formatting against its own `format`: in the C locale (`localeconv()` is C's, in this process), and
    # with `,` grouping by three, which formats as a locale grouping by three does
    assert locale.localeconv()["grouping"] == [] and under({"locale": locale.localeconv()})(
        P.joinedstr([fv(x, None, ("n",))]), {"x": 1234}).evaluate() == "1234"
    commas = {"decimal_point": ".", "thousands_sep": ",", "grouping": [3, 0]}
    for bound, specs in [(1234567, [("n", ",d"), ("012n", "012,d"), ("+n", "+,d"), ("*^15n", "*^15,d")]),
                         (-12345.678, [("n", ",g"), (".9n", ",.9g"), ("#012n", "#012,g"), ("-n", "-,g")]),
                         (1234.5 - 1e7j, [("n", ",g"), (">24n", ">24,g")])]:
        for spec, comma in specs:
            field = P.joinedstr([fv(x, None, (spec,))])
            assert under({"locale": c})(field, {"x": bound}).evaluate() == format(bound, spec), spec
            assert under({"locale": commas})(field, {"x": bound}).evaluate() == format(bound, comma), spec
    # What Python refuses stays, named, under a locale too (PYS-15): a str's `n`, and `n` with `,`
    for bound, spec, owner in [("ab", "n", "str"), (1, ",n", "int")]:
        assert [r.reason for r in under({"locale": de})(P.joinedstr([fv(x, None, (spec,))]), {"x": bound}).residues] == [
            f"Python3 joinedstr's format spec {spec!r} of {owner} is not simplified"]
    for value, message in [
        ("de", "the option 'locale' is 'stay' or a mapping of 'decimal_point', 'thousands_sep' and 'grouping', not 'de'"),
        *[(bad, "the option 'locale' maps 'decimal_point' and 'thousands_sep' to strs, and 'grouping' to a list of ints from 0 to 127")
          for bad in [{"decimal_point": ","}, {**de, "thousands_sep": 1}, {**de, "grouping": "3"}, {**de, "grouping": ["3"]},
                      {**de, "grouping": [1.5]}, {**de, "grouping": [-1]}, {**de, "grouping": [128]}]],
    ]:
        assert refusal(lambda: under({"locale": value})(x)) == message, value

<a id="PYS-19"></a>
## PYS-19 · A field whose text is the implementation's: the scope's option `text` "cpython" writes it as CPython writes it, where this implementation can

In [ ]:
with store.transaction():
    # Under "cpython", a template's text is its `repr`, as CPython 3.14 writes it, and a format spec of a value whose
    # `__format__` is `object`'s raises its TypeError, named as CPython names its type; by default they stay, named (PYS-04)
    texts = under({"text": "cpython"})
    tstring = P.templatestr(["a", P.interpolation(x, None, None, False, "x")])
    shown = "Template(strings=('a', ''), interpolations=(Interpolation(1, 'x', None, ''),))"
    for spec, written in [(P.joinedstr([fv(tstring, "r")]), shown), (P.joinedstr([fv(tstring)]), shown),
                          (P.joinedstr([fv(P.subscript(P.attribute(tstring, "interpolations"), 0), "a")]), "Interpolation(1, 'x', None, '')")]:
        assert texts(spec, {"x": 1}).evaluate() == written, source(spec)
    assert reasons(P.joinedstr([fv(tstring)]), {"x": 1}) == [("Python3 joinedstr's field of Template is not simplified", "a")]
    for spec, variables, name in [(P.joinedstr([fv(tstring, None, (">5",))]), {"x": 1}, "string.templatelib.Template"),
                                  (P.joinedstr([fv(P.subscript(P.attribute(tstring, "interpolations"), 0), None, (">5",))]), {"x": 1},
                                   "string.templatelib.Interpolation"),
                                  (P.joinedstr([fv(x, None, (">5",))]), {"x": {1}}, "set"),
                                  (P.joinedstr([fv(P.lambda_(1), None, (">5",))]), {}, "function"),
                                  (P.joinedstr([fv(P.generatorexp(n("p"), comp("p", v)), None, (">5",))]), {"v": [1]}, "generator")]:
        with raises(TypeError, match=f"unsupported format string passed to {name}.__format__"):
            texts(spec, variables)
    # A set's text, which is its hash table's order, and an object's, which is its address: Python writes CPython's text
    # (where TypeScript cannot, which no JavaScript value has, and they stay, named); a host object's `__format__` is its own
    assert texts(P.joinedstr([fv(x)]), {"x": {2, 1}}).evaluate() == str({2, 1})
    assert re.fullmatch(r"<function <lambda> at 0x[0-9a-f]+>", texts(P.joinedstr([fv(P.lambda_(1))])).evaluate())
    assert re.fullmatch(r"<generator object <genexpr> at 0x[0-9a-f]+>",
                        texts(P.joinedstr([fv(P.generatorexp(n("p"), comp("p", v)), "r")]), {"v": [1]}).evaluate())
    with raises(TypeError, match="unsupported format string passed to object.__format__"):
        texts(P.joinedstr([fv(x, None, (">5",))]), {"x": object()})
    with raises(ValueError, match="the option 'text' is 'stay' or 'cpython', not 'cp'"):
        under({"text": "cp"})(x)

<a id="PYS-20"></a>
## PYS-20 · A parameter assigned a value folds by Python's rules; one neither assigned nor bound stays where it stood, named, and Python's short circuits fold it away

In [ ]:
with store.transaction():
    n = store.References.OfParameter(lambda p: p.name("n"))
    assigned = lambda value: store.Contexts.OfContext(lambda c: c.parameters({  # noqa: E731
        "n": lambda p: p.name("n").value(lambda v: v.value(value))}))
    # Assigned a value, a parameter folds by Python's rules, its value lifted as a variable's is
    assert PV.OfAny(term(P.binop("+", n, 1)), assigned(2)).evaluate() == 3
    assert PText.ToText(term(PV.OfAny(term(P.binop("*", n, P.name("x"))), assigned("ab")).residual)) == "'ab' * x"
    assert PV.OfAny(term(P.binop("+", P.name("n"), n)), PV.Scope({"n": 10}).within(assigned(1))).evaluate() == 11
    # Neither assigned nor bound, it stays where it stood, named, while its sibling folds around it
    kept = PV.OfAny(term(P.binop("+", n, P.binop("*", 2, 3))))
    assert kept.parameters == {"n"} and F.same(term(kept.residual), term(P.binop("+", n, 6)))
    # An ancestor folds it away where Python's short circuit never evaluates it
    assert PV.OfAny(term(P.boolop("and", False, n))).evaluate() is False and PV.OfAny(term(P.boolop("or", 1, n))).evaluate() == 1
    assert PV.OfAny(term(P.boolop("and", True, n))).parameters == {"n"}
    # In an f-string or a t-string, a field's parameter is named while its context neither assigns nor binds it
    fields = lambda field: [field(n), field(P.binop("+", 1, 1))]  # noqa: E731
    assert PV.OfAny(term(P.joinedstr(fields(P.formattedvalue)))).parameters == {"n"}
    assert PV.OfAny(term(P.templatestr(fields(P.interpolation)))).parameters == {"n"}
    assert PV.OfAny(term(P.templatestr(fields(P.interpolation))), assigned(2)).parameters == set()

<a id="PYS-21"></a>
## PYS-21 · A type parameter in a lambda's annotation: Python never evaluates it, so a call folds either way, and the lambda that stays holds the type assigned in its place, or the parameter, named

In [ ]:
from mbse.Expressions.Python.Types import Expressions as PT

with store.transaction():
    T = store.References.OfParameter(lambda p: p.name("T"))
    typed = lambda spec: F.term(store.Expressions.Framework.Expression(lambda e: e.root(PT.DIALECT.spec(spec))))  # noqa: E731
    assigned = lambda held: store.Contexts.OfContext(lambda c: c.parameters({  # noqa: E731
        "T": lambda p: p.name("T").value(lambda v: v.value(held))}))
    annotated = P.lambda_(P.binop("+", P.name("x"), 1), [P.arg("x", annotation=T)])  # a Python Types term's place
    # Python never evaluates an annotation: a call folds whatever its type parameter is
    assert PV.OfAny(term(P.call(annotated, 2))).evaluate() == 3
    assert PV.OfAny(term(P.call(annotated, 2)), assigned(typed(P.name("int")))).evaluate() == 3
    # The lambda, a closure with no closed constructor expression, stays: its annotation the type assigned, read in the
    # parameter's place, or the parameter, named, where its context neither assigns nor binds it
    kept = PV.OfAny(term(annotated))
    assert kept.parameters == {"T"} and F.same(term(kept.residual), term(annotated))
    given = PV.OfAny(term(annotated), assigned(typed(P.name("int"))))
    written = P.lambda_(P.binop("+", P.name("x"), 1), [P.arg("x", annotation=P.name("int"))])
    assert given.parameters == set() and F.same(term(given.residual), term(written))